# 第11回 プロンプトと RAG — 書き方を変えるだけで、何がどこまで変わるのか

**このノートブックは第11回だけで完結します。** プログラミングの経験は要りません。

やること:
1. 同じ質問を6通りの書き方で聞き、答えがどれだけ変わるかを測る
2. RAG(探す → 渡す → 答えさせる)を自分で組む
3. **答えが資料にない質問をして、何が起きるかを見る**
4. **正反対の文書を取り違えることを、自分で再現する**

所要時間:実習 約25分 / 課題 約40分

> **注意**:3GBほどのモデルをダウンロードします。時間に余裕をもって実行してください。

## 0. 準備

上から順にセルを実行してください(セルを選んで **Shift + Enter**)。

> **注意**:3GBほどのモデルをダウンロードします。最初の2セルで3〜5分かかります。

In [ ]:
# 準備(このセルを最初に1回だけ実行。3〜5分かかります)
!pip install -q transformers sentence-transformers japanize-matplotlib
import warnings; warnings.filterwarnings("ignore")
import torch, numpy as np
import matplotlib.pyplot as plt, japanize_matplotlib
from transformers import AutoTokenizer, AutoModelForCausalLM
from sentence_transformers import SentenceTransformer
plt.rcParams["font.size"] = 12
print("準備できました")

In [ ]:
# 2つのモデルを読み込みます(初回は3GBほどダウンロードします)
#   ① 文章を作るモデル(GPT-2 large)
#   ② 文の意味を数字にするモデル(第7回で使ったもの)
生成 = AutoModelForCausalLM.from_pretrained("gpt2-large"); 生成.eval()
トークナイザ = AutoTokenizer.from_pretrained("gpt2")
埋め込み = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")

def 数字にする(文リスト):
    v = 埋め込み.encode(文リスト)
    return v / np.linalg.norm(v, axis=1, keepdims=True)

def 次の語(文, 上位=3):
    ids = トークナイザ.encode(文, return_tensors="pt")
    with torch.no_grad():
        p = torch.softmax(生成(ids).logits[0, -1], dim=-1)
    上 = torch.topk(p, 上位)
    return [(トークナイザ.decode([i]).strip() or "␣", float(v) * 100)
            for i, v in zip(上.indices, 上.values)]

print("読み込み完了")

## 1. 書き方だけで、どれだけ変わるか

**同じモデル・同じ知識**に、同じことを6通りの書き方で聞きます。
変えるのは文の書き方だけです。

In [ ]:
# ===== ① 書き方だけで、どれだけ変わるか =====
# 同じことを聞いています。モデルも知識も同じ。変えるのは書き方だけ。
言い方 = [
    "The capital of France is",
    "France's capital is",
    "The capital city of France is called",
    "Q: What is the capital of France? A:",
    "Question: What is the capital of France?\nAnswer: The capital of France is",
    "Paris is the capital of France. The capital of France is",
]
記録 = []
print(f"{'1位の語':<10}{'確率':>8}   書き方")
print("-" * 78)
for 文 in 言い方:
    r = 次の語(文)
    記録.append((文, r[0][0], r[0][1]))
    印 = "◎" if r[0][0] == "Paris" else "×"
    表示 = 文.replace("\n", " / ")
    print(f"{r[0][0]:<10}{r[0][1]:>7.1f}% {印} {表示[:58]}")

print()
print("同じモデル・同じ知識です。それでも、答えが出るかどうかが変わります。")
print("このモデルは『フランスの首都』を知らないのではありません。")
print("聞き方によって、取り出せたり取り出せなかったりするのです。")

In [ ]:
# グラフで見る
名前 = ["文の途中で切る", "所有格で切る", "…is called", "Q: / A:",
        "Question: / Answer:", "答えを先に書く"]
値 = [r[2] for r in 記録]
色 = ["#E08A2B" if r[1] != "Paris" else "#3A46A8" for r in 記録]

plt.figure(figsize=(8.5, 4))
plt.barh(range(len(値)), 値, color=色)
plt.yticks(range(len(値)), 名前); plt.gca().invert_yaxis()
plt.xlabel("正解 Paris が1位に来たときの確率(%)")
plt.title("オレンジ = 正解が出なかった書き方")
plt.grid(alpha=.3, axis="x"); plt.tight_layout(); plt.show()

print("いちばん強いのは『答えを先に書く』でした。")
print("これはほとんど答えを渡しているようなものですが ── それが次の RAG の原理です。")

> **プロンプトは「命令」ではありません**
>
> モデルがしているのは、第1回からずっと同じ「次の語を確率で選ぶ」だけです。
> プロンプトとは、**どういう続きがありそうな文脈を作るか**という作業です。
>
> 「Q:」「A:」の形が効くのは、その形の文章が学習データに大量にあったからです。

## 2. RAG を組む — 探す → 渡す → 答えさせる

モデルが知らないことは答えられません。ならば、**答えが書いてある資料を一緒に渡せばよい**。

- **① 探す** … 第7回の埋め込みで、質問に近い文書を選ぶ
- **② 渡す** … 選んだ文書を質問の前に貼る(学習しません。重みは変わりません)
- **③ 答えさせる** … 目の前の文章の続きとして自然な語を選ばせる

モデルが**絶対に知らない架空の事実**で試します。

In [ ]:
# ===== ② RAG を組む =====
# モデルが絶対に知らない「架空の事実」を8件用意します。
文書 = [
    "The tuition fee at Kelmore University is 42,000 yen per semester.",
    "Kelmore University was founded in 1954 by a group of local merchants.",
    "The library at Kelmore University closes at nine in the evening.",
    "Vantis Corporation was founded by Reiko Amano in 2003.",
    "Vantis Corporation employs about 1,200 people in three countries.",
    "The Blaythorne Bridge was completed in 1987 after six years of work.",
    "The Blaythorne Bridge carries about 40,000 vehicles every day.",
    "Parking at Kelmore University costs 300 yen per hour for visitors.",
]
文書ベクトル = 数字にする(文書)

def 探す(質問, 件数=2):
    """① 探す:質問にいちばん近い文書を返す(第7回の埋め込み)"""
    q = 数字にする([質問])[0]
    近さ = 文書ベクトル @ q
    順 = np.argsort(-近さ)
    return [(文書[i], float(近さ[i])) for i in 順[:件数]]

def RAGで答える(質問, 続き):
    """② 渡す + ③ 答えさせる"""
    取れた, 近さ = 探す(質問, 1)[0]
    無し = 次の語(続き)                    # 資料なし
    有り = 次の語(取れた + " " + 続き)      # 資料あり
    print(f"■ {質問}")
    print(f"   取り出した文書(近さ {近さ:.3f}): {取れた}")
    print(f"   RAG なし → " + ", ".join(f"{w}({v:.1f}%)" for w, v in 無し))
    print(f"   RAG あり → " + ", ".join(f"{w}({v:.1f}%)" for w, v in 有り))
    print()
    return 近さ

print("=== RAG あり / なし ===\n")
質問集 = [
    ("What is the tuition fee at Kelmore University?", "The tuition fee at Kelmore University is"),
    ("Who founded Vantis Corporation?", "Vantis Corporation was founded by"),
    ("When was the Blaythorne Bridge completed?", "The Blaythorne Bridge was completed in"),
]
正しい近さ = [RAGで答える(q, c) for q, c in 質問集]

## 3. RAG が壊れるとき ① — 答えが資料にない

検索は「いちばん近い1件」を返します。
では、**答えがどこにも書かれていない質問**をしたら、どうなるでしょうか。

In [ ]:
# ===== ③ 答えが資料にない質問をしてみる =====
print("=== 答えが8件の文書のどこにも書かれていない質問 ===\n")
失敗質問 = [
    ("How much does parking cost at Vantis Corporation?", "Parking at Vantis Corporation costs"),
    ("When was Kelmore University's library built?", "The library at Kelmore University was built in"),
]
誤った近さ = []
for q, c in 失敗質問:
    上位 = 探す(q, 2)
    答 = 次の語(上位[0][0] + " " + c)
    誤った近さ.append(上位[0][1])
    print(f"■ {q}")
    print(f"   1位(近さ {上位[0][1]:.3f}): {上位[0][0]}")
    print(f"   2位(近さ {上位[1][1]:.3f}): {上位[1][0]}")
    print(f"   → " + ", ".join(f"{w}({v:.1f}%)" for w, v in 答))
    print(f"   ※ 正しい答えは、資料のどこにも存在しません\n")

print("検索は「該当なし」とは言いません。必ず、いちばん近い1件を返します。")
print("そしてモデルは、渡された文書を忠実に使って答えます。")
print()
print("=== では「近さ」で足切りすれば防げるのか ===")
print(f"  正しく取れたとき: " + " / ".join(f"{x:.3f}" for x in 正しい近さ))
print(f"  間違えたとき    : " + " / ".join(f"{x:.3f}" for x in 誤った近さ))
print()
print(f"  誤った取得の最大 {max(誤った近さ):.3f} が、正しい取得の最小 {min(正しい近さ):.3f} より高い。")
print("  → 足切り線を引くと、正しい取得を捨てて、誤った取得を通してしまいます。")
print("     近さのスコアは、正しさを教えてくれません。")

## 4. RAG が壊れるとき ② — 正反対の文書を取り違える

第7回で、「この薬は安全です」と「この薬は危険です」の近さが **0.765** になるのを見ました。
正反対の意味なのに、ほぼ同じ扱いです。

その弱点は、RAG ではどう表に出るでしょうか。

In [ ]:
# ===== ④ 正反対の文書を、取り違える(第7回の続き)=====
薬 = [
    "この薬は妊娠中でも安全に服用できます。",
    "この薬は妊娠中には服用できません。重篤な副作用の報告があります。",
    "この薬は食後に服用してください。",
    "この薬は1日3回まで服用できます。",
]
薬ベクトル = 数字にする(薬)

質問たち = ["妊娠中に飲んでも大丈夫ですか", "妊娠中は飲めますか", "妊娠中の服用は避けるべきですか"]
差の記録 = []
for q in 質問たち:
    近さ = 薬ベクトル @ 数字にする([q])[0]
    順 = np.argsort(-近さ)
    差 = float(近さ[順[0]] - 近さ[順[1]])
    差の記録.append(差)
    print(f"■ 質問:「{q}」")
    print(f"   1位(近さ {近さ[順[0]]:.3f}): {薬[順[0]]}")
    print(f"   2位(近さ {近さ[順[1]]:.3f}): {薬[順[1]]}")
    print(f"   差: {差:.3f}\n")

print(f"3つの質問での差: " + " / ".join(f"{d:.3f}" for d in 差の記録))
print("正反対の2件が、つねに 0.02 以内に並びます。")
print("どちらが1位になるかは、ほとんど偶然です。")
print()
print("第7回で「安全です / 危険です」の近さが 0.765 になるのを見ました。")
print("埋め込みは『一緒に出てくるか』を学んだもので、意味を理解してはいません。")
print("その弱点が、RAG では「正反対の資料を渡す」という形で表に出ます。")

> **今日いちばん持ち帰ってほしいこと**
>
> RAG は「間違いを減らす」技術であって、「なくす」技術ではありません。
> 変わったのは**間違いの出どころ**です ── モデルの記憶から、検索の取り違えへ。
>
> ただし1つ、確実に良くなったことがあります。**出典を示せる**ようになったことです。
> 「この文書にこう書いてある」と言えれば、人間が確かめられます。

## 5. 自分で試す

**壊しても大丈夫**です(開き直せば元に戻ります)。

In [ ]:
# ===== 自分で試す(課題用)=====
# (a) 自分で架空の文書を作って、RAG を動かす
私の文書 = [
    "The Aoba Circle meets every Tuesday at six in the clubroom.",
    "The Aoba Circle membership fee is 3,000 yen per year.",
    "The Aoba Circle was started in 2011 by four second-year students.",
    "The Midori Cafe closes at eight on weekdays and at six on Sundays.",
    "The Midori Cafe offers a student discount of 20 percent.",
]
私のベクトル = 数字にする(私の文書)

def 私のRAG(質問, 続き):
    q = 数字にする([質問])[0]
    近さ = 私のベクトル @ q
    i = int(np.argmax(近さ))
    print(f"■ {質問}")
    print(f"   取れた(近さ {近さ[i]:.3f}): {私の文書[i]}")
    print(f"   なし → " + ", ".join(f"{w}({v:.1f}%)" for w, v in 次の語(続き)))
    print(f"   あり → " + ", ".join(f"{w}({v:.1f}%)" for w, v in 次の語(私の文書[i] + " " + 続き)))
    print()

私のRAG("How much is the Aoba Circle membership fee?",
        "The Aoba Circle membership fee is")
私のRAG("What time does the Aoba Circle clubroom open?",     # ← 答えは資料にありません
        "The Aoba Circle clubroom opens at")

# (b) 自分で正反対の文書を作って、取り違えを再現する
私の対 = [
    "この施設はペットを連れて入場できます。",
    "この施設にはペットを連れて入場できません。",
    "この施設は年中無休で営業しています。",
]
対ベクトル = 数字にする(私の対)
for q in ["犬を連れて行けますか", "ペットの同伴は禁止ですか"]:
    近さ = 対ベクトル @ 数字にする([q])[0]
    順 = np.argsort(-近さ)
    print(f"「{q}」")
    print(f"   1位 {近さ[順[0]]:.3f}: {私の対[順[0]]}")
    print(f"   2位 {近さ[順[1]]:.3f}: {私の対[順[1]]}   (差 {近さ[順[0]]-近さ[順[1]]:.3f})")

## 6. 提出課題(manaba)

**締切:次回講義の前日 23:59**

### 課題1:RAG の答えを検算する
次の RAG の答えを検算しなさい。文書集合には下の3件**だけ**が入っています。

```
【質問】   「返品は何日以内ですか」
【AI の答え】「返品は14日以内です。(出典:規約 第8条)」

【文書集合】
  ① 第7条:配送は注文から3営業日以内に行う。
  ② 第8条:交換は商品到着後14日以内に限る。
  ③ 第9条:領収書は購入後1年間再発行できる。
```

- (a) この答えは正しいですか
- (b) なぜこの間違いが起きたと考えられますか
- (c) この AI は、代わりに何と答えるべきでしたか

### 課題2:ノートブックを動かして報告する
- (a) 同じ質問の書き方6通り以上の表(1位の語と確率)
- (b) 自分で作った架空文書5件での RAG あり/なしの比較表
  **必ず「答えが資料にない質問」を1つ含めて、何が返ったかを書くこと**
- (c) 正反対の文書2件を自分で作り、質問3通りでの近さの差を表にする

### 課題3:次の主張を批判する(300字以上)

> 「社内規程チャットボットを RAG で作りました。必ず出典を示すので、
> でたらめを言うことはありません。これで問い合わせ対応は無人化できます。」

考える材料:
- 出典が示されていれば、答えが正しいと言えますか(課題1を思い出して)
- 規程に**書かれていない**ことを聞かれたら、何が起きますか
- 「〜してよい」と「〜してはならない」を、検索は区別できますか
- 「近さで足切りすれば安全」は成り立ちますか(実測値を根拠に)
- 「見つけること」と「解釈して回答すること」は同じ仕事ですか

**加点**:「では、どこまで任せられるか」まで書けると評価が上がります。

---

採点は、正解かどうかよりも **試したことと考えたこと** を見ます。
課題2(b)(c) は、うまくいったかではなく、**数字を挙げて説明できているか**を見ます。